In [ ]:
# Data Loading and Preprocessing:-

# Download dataset
!wget https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip

# Unzip dataset
!unzip master.zip

# Importing Libraries:-
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Path
train_path = "/content/PlantVillage-Dataset-master/raw/color"

# Preprocessing
img_size = 128
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    train_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    train_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

--2026-06-04 07:34:47--  https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/spMohanty/PlantVillage-Dataset/zip/refs/heads/master [following]
--2026-06-04 07:34:48--  https://codeload.github.com/spMohanty/PlantVillage-Dataset/zip/refs/heads/master
Resolving codeload.github.com (codeload.github.com)... 20.205.243.165
Connecting to codeload.github.com (codeload.github.com)|20.205.243.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘master.zip’

master.zip              [              <=>   ]  16.04M  5.14MB/s               ^C
Archive:  master.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  

FileNotFoundError: [Errno 2] No such file or directory: '/content/PlantVillage-Dataset-master/raw/color'

In [ ]:
# CNN Model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),

    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(train_data.num_classes, activation='softmax')
])

# Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

In [ ]:
# Model Evaluation


print("Training Accuracy:", history.history['accuracy'][-1])
print("Validation Accuracy:", history.history['val_accuracy'][-1])

In [ ]:
# Plotting Training Curves

import matplotlib.pyplot as plt

# Accuracy Graph
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Train', 'Validation'])

# Loss Graph
plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'])

plt.show()

In [ ]:
# Full Pipeline Execution:-

import os
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt

tomato_early_blight_dir = '/content/PlantVillage-Dataset-master/raw/color/Tomato___Early_blight/'

# Check if the directory exists and is not empty
if os.path.exists(tomato_early_blight_dir) and os.listdir(tomato_early_blight_dir):
    # Get the first image file from the directory
    filename = os.listdir(tomato_early_blight_dir)[0]
    img_path = os.path.join(tomato_early_blight_dir, filename)

    img = image.load_img(img_path, target_size=(128,128))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    prediction = model.predict(img_array)

    class_names = list(train_data.class_indices.keys())

    predicted_class = class_names[np.argmax(prediction)]
    confidence = np.max(prediction)

    print("Prediction:", predicted_class)
    print("Confidence:", confidence)

    # Decision support
    solutions = {
        "Tomato__Early_blight": "Remove infected leaves",
        "Tomato__Late_blight": "Use fungicide spray",
        "Healthy": "No action needed"
    }

    if predicted_class in solutions:
        print("Solution:", solutions[predicted_class])

    # Image Show
    plt.imshow(img)
    plt.title(predicted_class)
    plt.axis('off')
    plt.show()

else:
    print(f"Directory not found or is empty: {tomato_early_blight_dir}")
    img_path = None  # Set to None or a default image path if desired